In [396]:
import pandas as pd

In [397]:
df = pd.read_csv("healthcare_dataset.csv")

In [398]:
df.head()

,Name,Age,Gender,Blood Type,Medical Condition,Date of Admission,Doctor,Hospital,Insurance Provider,Billing Amount,Room Number,Admission Type,Discharge Date,Medication,Test Results
0,Bobby JacksOn,30,Male,B-,Cancer,2024-01-31,Matthew Smith,Sons and Miller,Blue Cross,18856.281306,328,Urgent,2024-02-02,Paracetamol,Normal
1,LesLie TErRy,62,Male,A+,Obesity,2019-08-20,Samantha Davies,Kim Inc,Medicare,33643.327287,265,Emergency,2019-08-26,Ibuprofen,Inconclusive
2,DaNnY sMitH,76,Female,A-,Obesity,2022-09-22,Tiffany Mitchell,Cook PLC,Aetna,27955.096079,205,Emergency,2022-10-07,Aspirin,Normal
3,andrEw waTtS,28,Female,O+,Diabetes,2020-11-18,Kevin Wells,"Hernandez Rogers and Vang,",Medicare,37909.782410,450,Elective,2020-12-18,Ibuprofen,Abnormal
4,adrIENNE bEll,43,Female,AB+,Cancer,2022-09-19,Kathleen Hanna,White-White,Aetna,14238.317814,458,Urgent,2022-10-09,Penicillin,Abnormal


In [399]:
df.columns

Index(['Name', 'Age', 'Gender', 'Blood Type', 'Medical Condition',
       'Date of Admission', 'Doctor', 'Hospital', 'Insurance Provider',
       'Billing Amount', 'Room Number', 'Admission Type', 'Discharge Date',
       'Medication', 'Test Results'],
      dtype='str')

In [400]:
df = df.drop(columns = ["Name", "Hospital", "Doctor", "Room Number"]) # noise creators

In [401]:
df.dtypes

Age                     int64
Gender                    str
Blood Type                str
Medical Condition         str
Date of Admission         str
Insurance Provider        str
Billing Amount        float64
Admission Type            str
Discharge Date            str
Medication                str
Test Results              str
dtype: object

In [402]:
import datetime

In [403]:
df["Date of Admission"] = pd.to_datetime(df["Date of Admission"])
df["Discharge Date"] = pd.to_datetime(df["Discharge Date"])

In [404]:
df["Stay Duration"] = (
    df["Discharge Date"] - df["Date of Admission"]
).dt.days

In [405]:
df.head()

,Age,Gender,Blood Type,Medical Condition,Date of Admission,Insurance Provider,Billing Amount,Admission Type,Discharge Date,Medication,Test Results,Stay Duration
0,30,Male,B-,Cancer,2024-01-31,Blue Cross,18856.281306,Urgent,2024-02-02,Paracetamol,Normal,2
1,62,Male,A+,Obesity,2019-08-20,Medicare,33643.327287,Emergency,2019-08-26,Ibuprofen,Inconclusive,6
2,76,Female,A-,Obesity,2022-09-22,Aetna,27955.096079,Emergency,2022-10-07,Aspirin,Normal,15
3,28,Female,O+,Diabetes,2020-11-18,Medicare,37909.782410,Elective,2020-12-18,Ibuprofen,Abnormal,30
4,43,Female,AB+,Cancer,2022-09-19,Aetna,14238.317814,Urgent,2022-10-09,Penicillin,Abnormal,20


In [406]:
df.columns

Index(['Age', 'Gender', 'Blood Type', 'Medical Condition', 'Date of Admission',
       'Insurance Provider', 'Billing Amount', 'Admission Type',
       'Discharge Date', 'Medication', 'Test Results', 'Stay Duration'],
      dtype='str')

In [407]:
df.dtypes

Age                            int64
Gender                           str
Blood Type                       str
Medical Condition                str
Date of Admission     datetime64[us]
Insurance Provider               str
Billing Amount               float64
Admission Type                   str
Discharge Date        datetime64[us]
Medication                       str
Test Results                     str
Stay Duration                  int64
dtype: object

In [408]:
df["Month of Admission"] = df["Date of Admission"].dt.month.astype("str")

In [409]:
df.head()

,Age,Gender,Blood Type,Medical Condition,Date of Admission,Insurance Provider,Billing Amount,Admission Type,Discharge Date,Medication,Test Results,Stay Duration,Month of Admission
0,30,Male,B-,Cancer,2024-01-31,Blue Cross,18856.281306,Urgent,2024-02-02,Paracetamol,Normal,2,1
1,62,Male,A+,Obesity,2019-08-20,Medicare,33643.327287,Emergency,2019-08-26,Ibuprofen,Inconclusive,6,8
2,76,Female,A-,Obesity,2022-09-22,Aetna,27955.096079,Emergency,2022-10-07,Aspirin,Normal,15,9
3,28,Female,O+,Diabetes,2020-11-18,Medicare,37909.782410,Elective,2020-12-18,Ibuprofen,Abnormal,30,11
4,43,Female,AB+,Cancer,2022-09-19,Aetna,14238.317814,Urgent,2022-10-09,Penicillin,Abnormal,20,9


# **Target**: *Billing Amount*

### **Numerical Features**:
- Standard Scaler
### **Categorical Features**:
- One-hot encode all
### **Feature selection**:
- Optional, but try using ANOVA

In [410]:
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.feature_selection import f_regression, SelectKBest
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error

In [411]:
df = df.drop(columns = ["Date of Admission", "Discharge Date"])

In [412]:
x = df.drop(columns = ["Billing Amount"])
y = df["Billing Amount"]

In [413]:
x.dtypes

Age                   int64
Gender                  str
Blood Type              str
Medical Condition       str
Insurance Provider      str
Admission Type          str
Medication              str
Test Results            str
Stay Duration         int64
Month of Admission      str
dtype: object

In [414]:
transformer = ColumnTransformer([
    ("scaler", StandardScaler(), make_column_selector(dtype_include=["int64"])),
    ("encoder", OneHotEncoder(drop = "first", sparse_output=False), make_column_selector(dtype_exclude="int64"))
])

In [415]:
pipeline = Pipeline([
    ("transform", transformer),
    #("select", SelectKBest(f_regression, k = 6)),
    ("model", LinearRegression())
])

In [416]:
x_train, x_test, y_train, y_test = train_test_split(
    x, y,
    test_size = 0.3,
    random_state = 11
)

In [417]:
models = [LinearRegression(), HistGradientBoostingRegressor(), RandomForestRegressor()]

In [418]:
k_vals = list(range(6, len(x.columns) + 1))

In [419]:
for model in models:
    pipeline.set_params(model = model)
    pipeline.fit(x_train, y_train)
    y_pred = pipeline.predict(x_test)
    print("-" * 50)
    print(f"R2: {r2_score(y_test, y_pred)}, MAE: {mean_absolute_error(y_test, y_pred)}")

--------------------------------------------------
R2: -0.0012213041829536309, MAE: 12305.737492604067
--------------------------------------------------
R2: -0.0007474783285257303, MAE: 12304.91867949199
--------------------------------------------------
R2: 0.05498513279281736, MAE: 11730.608820592874


In [420]:
rf = pipeline.named_steps["model"]

print(rf.feature_importances_)

[0.18618739 0.15867253 0.02949206 0.01797775 0.0173172  0.01629285
 0.01749618 0.01777847 0.01726111 0.01456442 0.01848603 0.01787601
 0.01911888 0.01978065 0.01636718 0.01934723 0.02156815 0.01996448
 0.02159033 0.02235946 0.02454036 0.01927905 0.0215585  0.02115979
 0.01885475 0.0231184  0.02443004 0.00711754 0.0143663  0.01556631
 0.01451497 0.01577845 0.01484932 0.01518197 0.01507925 0.01526348
 0.01553479 0.01430838]
